In [1]:
# Add the 'src' directory to the sys.path
# so that we can access our local modules
import sys
sys.path.append('../../src')

# Testing ElasticSearch for Literev Legal
- Indexing Data
- Querying Data

## Import libraries

In [2]:
import json
import os
import ssl

from elasticsearch import Elasticsearch

from etl import DataReader, DataNormalizer, DataTranslator

## Define constants

In [3]:
CA_CERT_PATH = "/home/felipe/dev/projects/poc-elasticsearch/containers/esconfig/certs/http_ca.crt"
JUDICIARY_SOURCE_DATA = "/home/felipe/dev/projects/judiciary-system/notebooks/data/output.jsonl"
SAMPLE_DATA = os.getcwd() + "/../../normalized-sample.jsonl"
ES_JUDICIARY_INDEX_NAME = "judiciary"

## Create elasticsearch client instance

In [4]:
ssl_context = ssl.create_default_context(cafile=CA_CERT_PATH)

es = Elasticsearch(
    ["https://localhost:9200"],
    basic_auth=("elastic", "worksfine"),
    ssl_context=ssl_context,
)

es

<Elasticsearch(['https://localhost:9200'])>

## Check elasticsearch connection HEALTH

In [5]:
es.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 1, 'active_shards': 1, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 1, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 50.0})

## Utility Functions

Helper functions for creating indexes, deleting indexes, indexing data, querying data from index and so on...

In [6]:
def index_json_data(es: Elasticsearch, file_path: str, index_name: str) -> None:
    """
    Reads data from a JSONL file and indexes it into Elasticsearch.

    Parameters:
    es (Elasticsearch): An Elasticsearch client instance.
    file_path (str): The path to the JSONL file.
    index_name (str): The name of the Elasticsearch index where data will be stored.
    """
    # Load JSON data from the file
    reader = DataReader(file_path)

    for _, document in reader.read_data():
        del document["document"]
        es.index(index=index_name, document=document, id=document["record_key"])


def prepare_document(doc: dict[str, str | list[dict[str, str]]]):
    """Normalize, clean and prepare document for indexing."""
    pass

def get_total_documents_in_index(es: Elasticsearch, index_name: str) -> int:
    """Returns total number of documents in es index"""

    response = es.count(
        index=index_name,
        body={
          "query": {
            "match_all": {},
          },
        },
    )

    return response["count"]

In [7]:
# delete index
es.options(ignore_status=[400,404]).indices.delete(index=ES_JUDICIARY_INDEX_NAME)

ObjectApiResponse({'acknowledged': True})

In [8]:
%%time
index_json_data(es=es, file_path=SAMPLE_DATA, index_name=ES_JUDICIARY_INDEX_NAME)

CPU times: user 322 ms, sys: 17.4 ms, total: 340 ms
Wall time: 3.1 s


In [9]:
get_total_documents_in_index(es=es, index_name=ES_JUDICIARY_INDEX_NAME)

200

In [10]:
es_query = {
    "query": {
        "bool": {
            "must": [
                {
                    "multi_match": {
                        "query": "brigandage",
                        "fields": ["document_text"],
                        "slop": 999,
                    }
                },
                # {
                #     "match": {
                #         "document_text": "brigandage"
                #     }
                # },
            ],
        }
    }
}


print("Total documents indexed:", get_total_documents_in_index(es=es, index_name=ES_JUDICIARY_INDEX_NAME))

response = es.search(
    index=ES_JUDICIARY_INDEX_NAME,
    body=es_query,
)

es_and_total = response["hits"]["total"]["value"]
print('query:brigandage Results --->', es_and_total)

Total documents indexed: 200
query:brigandage Results ---> 2
